# Librerías

In [2]:
import pandas as pd

# **Carga de datos**

In [3]:
# Carga del dataset desde github
url = "https://raw.githubusercontent.com/JeiGeek/Replica_clasificacion_multimodal_parkinson/main/requirements/MDS-UPDRS_Part_III_28Oct2025_estandarizado_pd_mbrs.csv"
df = pd.read_csv(url)

In [10]:
# Asignacion de tabla "grupo" para pacientes de parkinson
df["grupo"] = 1
df.head()

,PATNO,EVENT_ID,PDTRTMNT,PDSTATE,HRPOSTMED,HRDBSON,HRDBSOFF,PDMEDYN,DBSYN,ONOFFORDER,...,NP3RTCON,NP3TOT,DYSKPRES,DYSKIRAT,NHY,SEX,AGE_AT_VISIT,COHORT_DEFINITION,MBRS,grupo
0,3001,V17,1.0,1.0,1.0000,NaN,NaN,1.0,0.0,1.0,...,0.0,40.0,1.0,0.0,2.0,1,82.7,Parkinson's Disease,13.0,1
1,3002,V17,1.0,1.0,0.8333,NaN,NaN,1.0,0.0,1.0,...,0.0,31.0,1.0,0.0,4.0,0,82.7,Parkinson's Disease,14.0,1
2,3003,V21,1.0,1.0,1.2500,NaN,NaN,1.0,0.0,NaN,...,0.0,25.0,1.0,0.0,2.0,0,63.9,Parkinson's Disease,8.0,1
3,3005,SC,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,...,0.0,5.0,0.0,NaN,0.0,0,61.7,Parkinson's Disease,0.0,1
4,3006,V04,1.0,1.0,1.0000,NaN,NaN,1.0,0.0,NaN,...,0.0,34.0,0.0,NaN,2.0,0,56.7,Parkinson's Disease,14.0,1


In [ ]:
# porque se necesita colocar 1?????????

**Tabulares basados en:**
* Información Demográfica
* Evaluación Motora
* Movimientos de Manos
* Movimientos de Piernas
* Rigidez
* Temblor
* Puntuación Global
* Tratamiento y Estado Clínico
* Estimulación Cerebral Profunda (DBS)
* Discinesias
* Evaluaciones ON / OFF
* Estadificación de la Enfermedad

In [12]:
# Creación de descripciones de los nombres de cada una de las caracteristicas
descripciones_en = {
    'Edad': 'age', 'Genero': 'gender', 'NP3BRADY': 'bradykinesia',
    'NP3FACXP': 'reduced facial expression', 'NP3FRZGT': 'freezing of gait',
    'NP3FTAPL': 'left foot tapping', 'NP3FTAPR': 'right foot tapping',
    'NP3GAIT': 'gait impairment', 'NP3HMOVL': 'left hand movement',
    'NP3HMOVR': 'right hand movement', 'NP3KTRML': 'left hand kinesia',
    'NP3KTRMR': 'right hand kinesia', 'NP3LGAGL': 'left leg agility',
    'NP3LGAGR': 'right leg agility', 'NP3POSTR': 'posture',
    'NP3PRSPL': 'left hand postural tremor', 'NP3PRSPR': 'right hand postural tremor',
    'NP3PSTBL': 'postural instability', 'NP3PTRML': 'left hand resting tremor',
    'NP3PTRMR': 'right hand resting tremor', 'NP3RIGLL': 'left leg rigidity',
    'NP3RIGLU': 'left arm rigidity', 'NP3RIGN': 'neck rigidity',
    'NP3RIGRL': 'right leg rigidity', 'NP3RIGRU': 'right arm rigidity',
    'NP3RISNG': 'difficulty rising from chair', 'NP3RTALJ': 'left leg action tremor',
    'NP3RTALL': 'left leg action tremor', 'NP3RTALU': 'left arm action tremor',
    'NP3RTARL': 'right leg action tremor', 'NP3RTARU': 'right arm action tremor',
    'NP3RTCON': 'tremor constancy', 'NP3SPCH': 'speech impairment',
    'NP3TOT': 'MDS-UPDRS III total score', 'NP3TTAPL': 'left hand tapping',
    'NP3TTAPR': 'right hand tapping', 'DBSYN': 'deep brain stimulation symptoms',
    'DYSKPRES': 'dyskinesia presence', 'MBRS': 'dyskinesia scale',
    'H & Y Stage': 'Hoehn & Yahr stage', 'NHY': 'numeric Hoehn & Yahr stage',
    'DBSOFFTM': 'time DBS switched off', 'DBSONTM': 'time DBS switched on',
    'HRDBSOFF': 'hours from DBS off to NUPDRS3 exam',
    'HRDBSON': 'hours from DBS on to NUPDRS3 exam',
    'HRPOSTMED': 'hours from last PD medication to NUPDRS3 exam',
    'OFFEXAM': 'OFF exam performed', 'OFFNORSN': 'reason OFF exam not performed',
    'ONEXAM': 'ON exam performed', 'ONNORSN': 'reason ON exam not performed',
    'ONOFFORDER': 'order of OFF/ON Part III exams', 'PDMEDYN': 'Parkinson medication',
    'PDSTATE': 'current functional state', 'PDTRTMNT': 'treatment received',
    'Pat_id': 'patient ID', 'grupo': 'patient group'
}

# **Generación de prompts**

In [23]:
# Función de creación de prompts
def crear_prompts_parkinson(df, descripciones, idioma="es"):
    prompts = []
    for _, row in df.iterrows():

        # identificacion del idioma
        if idioma == "es":
          tipo_paciente = "paciente con Parkinson" if row['grupo'] == 1 else "control"
        else:
          tipo_paciente = "Parkinson's patient" if row['grupo'] == 1 else "control subject"

        prompt_parts = [f"{tipo_paciente}:"]

        # Construcción de prompots y tratamiento de valores nulos
        for col in df.columns:
            valor = row[col]

            if pd.isna(valor):
                valor = "no informado" if idioma == "es" else "not reported"
            else:
                if col == 'Genero':
                    if idioma == "es":
                        valor = "masculino" if valor == 1 else "femenino"
                    else:
                        valor = "male" if valor == 1 else "female"

            if col in descripciones:
                desc = descripciones[col]
                prompt_parts.append(f"{desc}: {valor}")
            else:
                prompt_parts.append(f"{col}: {valor}")

        prompts.append(". ".join(prompt_parts))
    return prompts

In [24]:
# Guardar prompts en inglés en columnas del DataFrame
df["prompts_400_ing"] = crear_prompts_parkinson(df, descripciones_en, idioma="en")

# Ver primeros 3 pacientes
for i in range(3):
    print(f"\nPaciente {i+1} (EN):\n{df['prompts_400_ing'].iloc[i]}")


Paciente 1 (EN):
Parkinson's patient:. PATNO: 3001. EVENT_ID: V17. treatment received: 1.0. current functional state: 1.0. hours from last PD medication to NUPDRS3 exam: 1.0. hours from DBS on to NUPDRS3 exam: not reported. hours from DBS off to NUPDRS3 exam: not reported. Parkinson medication: 1.0. deep brain stimulation symptoms: 0.0. order of OFF/ON Part III exams: 1.0. OFF exam performed: not reported. reason OFF exam not performed: not reported. DBSOFFYN: not reported. time DBS switched off: not reported. ON exam performed: 1.0. reason ON exam not performed: not reported. HIFUYN: not reported. DBSONYN: not reported. time DBS switched on: not reported. speech impairment: 2.0. reduced facial expression: 1.0. neck rigidity: 3.0. right arm rigidity: 1.0. left arm rigidity: 2.0. right leg rigidity: 1.0. left leg rigidity: 2.0. right foot tapping: 3.0. left foot tapping: 2.0. right hand movement: 3.0. left hand movement: 2.0. right hand postural tremor: 2.0. left hand postural tremor: 

In [26]:
# Generacion de archivo csv de los prompts
df[['PATNO', 'grupo', 'prompts_400_ing']].to_csv(
    "prompts_400_ing.csv",
    index=False,
    encoding='utf-8'
)

# **Extracción de embeddings**

# **Experimentación**